# Tarea 1 — LendingClub: revisión de resultados locales

Estado: revisión integrada al Jupyter Book local; pendiente de cierre académico y publicación.

## Datos y evaluación
CSV real completo identificado mediante SHA-256. Tras seleccionar los estados
Fully Paid y Charged Off, se utilizaron 1.345.310 préstamos: 1.076.248 de
entrenamiento y 269.062 de prueba. No se submuestreó el entrenamiento supervisado.
Ambos motores utilizaron la misma partición y los tres folds compartidos.
Las transformaciones aprendidas se ajustaron dentro de cada entrenamiento.

La partición es aleatoria estratificada por filas. No demuestra independencia
por prestatario ni capacidad de generalización temporal.

## Resultados medidos
Las familias de boosting obtuvieron el mayor AUC de validación en cada motor.
La selección se realizó por CV, no por el AUC de prueba.

### Discriminación por modelo

| Motor | Modelo | AUC CV | AUC prueba |
| --- | --- | --- | --- |
| sklearn | NB | 0.6820 | 0.6814 |
| sklearn | LR | 0.7073 | 0.7072 |
| sklearn | DT | 0.7009 | 0.7028 |
| sklearn | Boosting | 0.7152 | 0.7156 |
| sklearn | SVM | 0.5349 | 0.5709 |
| sklearn | RF | 0.7112 | 0.7117 |
| Spark | NB | 0.6821 | 0.6815 |
| Spark | LR | 0.7073 | 0.7072 |
| Spark | DT | 0.7009 | 0.7020 |
| Spark | Boosting | 0.7142 | 0.7149 |
| Spark | SVM | 0.6582 | 0.6463 |
| Spark | RF | 0.7102 | 0.7104 |
| sklearn | SVM cuadrática | 0.7059 | 0.7059 |

### Clasificación con el punto de decisión utilizado

| Motor | Modelo | Sensibilidad | Especificidad | Precisión | F1 |
| --- | --- | --- | --- | --- | --- |
| sklearn | NB | 0.1626 | 0.9472 | 0.4342 | 0.2365 |
| sklearn | LR | 0.0506 | 0.9890 | 0.5342 | 0.0924 |
| sklearn | DT | 0.0814 | 0.9806 | 0.5118 | 0.1405 |
| sklearn | Boosting | 0.0671 | 0.9871 | 0.5641 | 0.1199 |
| sklearn | SVM | 0.0162 | 0.9958 | 0.4924 | 0.0314 |
| sklearn | RF | 0.0561 | 0.9891 | 0.5627 | 0.1021 |
| Spark | NB | 0.1607 | 0.9479 | 0.4350 | 0.2347 |
| Spark | LR | 0.0502 | 0.9891 | 0.5345 | 0.0918 |
| Spark | DT | 0.0660 | 0.9844 | 0.5128 | 0.1170 |
| Spark | Boosting | 0.0790 | 0.9840 | 0.5515 | 0.1383 |
| Spark | SVM | 0.0000 | 1.0000 | 0.0000 | 0.0000 |
| Spark | RF | 0.0530 | 0.9897 | 0.5620 | 0.0969 |
| sklearn | SVM cuadrática | 0.0253 | 0.9951 | 0.5613 | 0.0485 |

### Tiempos observados en segundos

| Motor | Modelo | CV + ajuste (s) | Predicción (s) |
| --- | --- | --- | --- |
| sklearn | NB | 7.2514 | 0.3338 |
| sklearn | LR | 25.9110 | 0.3584 |
| sklearn | DT | 52.8877 | 0.2489 |
| sklearn | Boosting | 184.7301 | 1.4663 |
| sklearn | SVM | 1407.4231 | 0.3215 |
| sklearn | RF | 455.3521 | 2.6064 |
| Spark | NB | 122.0675 | 4.6848 |
| Spark | LR | 261.3319 | 2.4970 |
| Spark | DT | 309.8565 | 3.2140 |
| Spark | Boosting | 1396.8329 | 4.9244 |
| Spark | SVM | 451.7804 | 2.9481 |
| Spark | RF | 5626.0417 | 48.2955 |
| sklearn | SVM cuadrática | 43.5364 | 0.2495 |

**Abreviaturas:** NB: Naive Bayes; LR: regresión logística; DT: árbol de decisión; RF: bosque aleatorio. Boosting corresponde a HistGradientBoostingClassifier en sklearn y GBTClassifier en Spark.

**Observaciones de las ejecuciones:**

- **sklearn / SVM:** Ajuste final y candidato ganador sin convergencia.
- **Spark / Boosting:** Implementación GBTClassifier.
- **Spark / SVM:** Todas las predicciones de prueba son clase 0.
- **sklearn / SVM cuadrática:** Variante adicional; pérdida diferente.

{download}`Descargar resultados completos <_static/tarea1_revision_mac/resumen_ejecuciones.csv>`

Los tiempos corresponden a ejecuciones observadas en el Mac, con configuraciones
diferentes por motor. No constituyen un benchmark con recursos equiparados y
réplicas temporales. CV_y_ajuste_s incluye búsqueda y ajuste final; no representa
únicamente el entrenamiento de un modelo.

La sensibilidad es baja en los puntos de decisión utilizados. Una exactitud
cercana al 80 % debe contextualizarse con el predominio de la clase negativa.
La SVM de Spark clasifica todos los registros de prueba como negativos:
su AUC evalúa el orden de las puntuaciones, no implica detección de positivos
con ese punto de decisión.

## Comparaciones estadísticas
Las evidencias incluyen AUC, diferencias pareadas e intervalos del 95 % mediante
DeLong, con ajuste de Holm sobre 38 comparaciones de AUC.
Los intervalos son individuales, no simultáneos. La inferencia está condicionada
a los modelos ajustados y supone independencia entre unidades evaluadas.
Los contrastes se presentan como exploratorios.

Una diferencia estadísticamente significativa no demuestra relevancia práctica;
un resultado no significativo tampoco demuestra equivalencia.
El margen histórico de 0,005 es ilustrativo.

Comparaciones completas: {download}`tabla CSV <_static/tarea1_revision_mac/contrastes/comparaciones_pareadas.csv>`.

HistGradientBoostingClassifier y GBTClassifier son implementaciones diferentes.
La variante SVM squared_hinge se documenta aparte; no sustituye silenciosamente
la ejecución original sin convergencia.

## EDA de entrenamiento
Revisión descriptiva posterior a los entrenamientos, sin utilizar prueba para
seleccionar nuevas transformaciones. No se presenta como análisis previo.

| variable | test | p | effect | effect_type | q_bh |
| --- | --- | --- | --- | --- | --- |
| loan_amnt | Mann Whitney | 0 | 0.101938 | biserial rangos | 0 |
| int_rate | Mann Whitney | 0 | 0.367389 | biserial rangos | 0 |
| annual_inc | Mann Whitney | 0 | -0.09714 | biserial rangos | 0 |
| dti | Mann Whitney | 0 | 0.153339 | biserial rangos | 0 |
| fico_range_high | Mann Whitney | 0 | -0.186688 | biserial rangos | 0 |
| open_acc | Mann Whitney | 4.56358e-174 | 0.0390774 | biserial rangos | 5.39333e-174 |
| revol_bal | Mann Whitney | 5.7098e-17 | -0.0116549 | biserial rangos | 5.7098e-17 |
| total_acc | Mann Whitney | 8.85307e-54 | -0.0214883 | biserial rangos | 9.59083e-54 |
| term | Chi cuadrado | 0 | 0.176261 | V de Cramer | 0 |
| grade | Chi cuadrado | 0 | 0.262093 | V de Cramer | 0 |
| home_ownership | Chi cuadrado | 0 | 0.0708506 | V de Cramer | 0 |
| verification_status | Chi cuadrado | 0 | 0.092472 | V de Cramer | 0 |
| purpose | Chi cuadrado | 0 | 0.055563 | V de Cramer | 0 |

Los valores p impresos como cero reflejan límites numéricos.
La magnitud de los efectos debe acompañar a la significación estadística.
Los efectos por rangos y V de Cramér no son directamente intercambiables.

DTI presenta 296 faltantes en entrenamiento, aproximadamente 0,0275 %.
Las categorías faltantes ya se habían convertido en Missing; el conteo de nulos
del archivo canónico no reconstruye por sí solo los faltantes originales.

Gráficos y tablas: carpeta evidencias/eda.

## Interpretabilidad local
Se explicaron los mismos casos en ambos motores:
falso positivo ID 509 y falso negativo ID 25.

| Motor | Caso | P modelo | LIME | Error absoluto | R² local |
| --- | --- | --- | --- | --- | --- |
| sklearn | 509 | 0.5336 | 0.4353 | 0.0983 | 0.5498 |
| sklearn | 25 | 0.2221 | 0.1969 | 0.0252 | 0.2961 |
| spark | 509 | 0.5331 | 0.4314 | 0.1017 | 0.4489 |
| spark | 25 | 0.1916 | 0.1988 | 0.0073 | 0.2807 |

En el falso positivo, ambas aproximaciones LIME quedan por debajo de 0,5,
aunque las probabilidades de los modelos superan ese valor. Por ello, estas
aproximaciones no reproducen fielmente la decisión en ese punto.

En el falso negativo, la aproximación puntual es más cercana, pero el R² local
es bajo. Los pesos describen aproximaciones locales; no son efectos causales
ni prueban estabilidad de las explicaciones.

Los HTML y pesos se conservan en evidencias/lime_sklearn y evidencias/lime_spark.

## Pendientes antes de declarar cerrada la entrega
- Contrastar la cobertura del informe con la guía específica de Tarea 1.
- Revisar disponibilidad de variables en el momento de predicción y alcance
  de las comprobaciones de duplicados o dependencia entre préstamos.
- Revisar las figuras y completar su interpretación.
- Completar la verificación de enlaces y preparar la publicación del libro.
- Documentar archivos y dependencias reproducibles antes de publicar.

Esta revisión corresponde exclusivamente a LendingClub. Las correcciones de
Tarea 2, Entregables 1 y 2 y el proyecto final de caries se mantienen separadas.


## Distribuciones numéricas por clase

![Distribuciones numéricas por clase](_static/tarea1_revision_mac/eda/figures/histograms.png)


### Vista complementaria de distribuciones

Los histogramas completos se conservan arriba. Esta vista utiliza un límite
común por variable: el percentil 99 calculado en todo el entrenamiento.
Cada intervalo muestra el porcentaje respecto al total no faltante de su clase;
la suma visible puede ser inferior al 100 % por los valores fuera del rango.
No se eliminaron registros ni se modificó el entrenamiento.

![Distribuciones centrales por clase](_static/tarea1_revision_mac/eda/figures/distribuciones_centrales.png)

{download}`Descargar límites y cantidades fuera de vista
<_static/tarea1_revision_mac/eda/figures/distribuciones_centrales.csv>`


## Distribución y dispersión por clase

![Distribución y dispersión por clase](_static/tarea1_revision_mac/eda/figures/boxplots.png)


## Correlación entre variables numéricas

![Correlación entre variables numéricas](_static/tarea1_revision_mac/eda/figures/correlation.png)


## Proporción de incumplimiento por categoría

![Proporción de incumplimiento por categoría](_static/tarea1_revision_mac/eda/figures/categories.png)


## Pesos locales LIME — sklearn

| row_id | tipo | condicion | peso_local_clase_1 |
| --- | --- | --- | --- |
| 509 | falso_positivo | int_rate > 15.99 | 0.126475 |
| 509 | falso_positivo | term= 60 months | 0.0758188 |
| 509 | falso_positivo | loan_amnt > 20000.00 | 0.048134 |
| 509 | falso_positivo | dti > 24.48 | 0.0456957 |
| 509 | falso_positivo | home_ownership=MORTGAGE | -0.0366376 |
| 509 | falso_positivo | annual_inc <= 45000.00 | 0.03586 |
| 509 | falso_positivo | 11208.50 < revol_bal <= 20110.25 | -0.0109906 |
| 509 | falso_positivo | 674.00 < fico_range_high <= 694.00 | 0.0106726 |
| 25 | falso_negativo | term= 36 months | -0.0826461 |
| 25 | falso_negativo | annual_inc <= 45000.00 | 0.0448087 |
| 25 | falso_negativo | loan_amnt <= 7943.75 | -0.0435829 |
| 25 | falso_negativo | home_ownership=RENT | 0.0400144 |
| 25 | falso_negativo | open_acc <= 8.00 | -0.0354889 |
| 25 | falso_negativo | fico_range_high > 714.00 | -0.0313455 |
| 25 | falso_negativo | total_acc <= 16.00 | 0.0256898 |
| 25 | falso_negativo | revol_bal <= 5914.00 | 0.0201607 |

- {download}`Descargar explicación HTML del caso 509 <_static/tarea1_revision_mac/lime_sklearn/lime_509.html>`
- {download}`Descargar explicación HTML del caso 25 <_static/tarea1_revision_mac/lime_sklearn/lime_25.html>`

## Pesos locales LIME — spark

| row_id | tipo | condicion | peso_local_clase_1 |
| --- | --- | --- | --- |
| 509 | falso_positivo | int_rate > 15.99 | 0.115763 |
| 509 | falso_positivo | term= 60 months | 0.0786353 |
| 509 | falso_positivo | loan_amnt > 20000.00 | 0.0473835 |
| 509 | falso_positivo | dti > 24.48 | 0.0437297 |
| 509 | falso_positivo | annual_inc <= 45000.00 | 0.0382391 |
| 509 | falso_positivo | home_ownership=MORTGAGE | -0.0363759 |
| 509 | falso_positivo | 11208.50 < revol_bal <= 20110.25 | -0.0128211 |
| 509 | falso_positivo | 674.00 < fico_range_high <= 694.00 | 0.0115002 |
| 25 | falso_negativo | term= 36 months | -0.0843389 |
| 25 | falso_negativo | annual_inc <= 45000.00 | 0.0451025 |
| 25 | falso_negativo | loan_amnt <= 7943.75 | -0.0433043 |
| 25 | falso_negativo | home_ownership=RENT | 0.0400692 |
| 25 | falso_negativo | open_acc <= 8.00 | -0.0380816 |
| 25 | falso_negativo | fico_range_high > 714.00 | -0.036807 |
| 25 | falso_negativo | total_acc <= 16.00 | 0.0314676 |
| 25 | falso_negativo | revol_bal <= 5914.00 | 0.0210794 |

- {download}`Descargar explicación HTML del caso 509 <_static/tarea1_revision_mac/lime_spark/lime_509.html>`
- {download}`Descargar explicación HTML del caso 25 <_static/tarea1_revision_mac/lime_spark/lime_25.html>`